# Core Module Testing

**Purpose**: Test the new `bmcs_cross_section.core` module functionality

**Status**: Development notebook - will be archived after Phase 1

**Date**: January 10, 2026

## Setup

Import the new core module and test basic functionality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import cached_property

# Import new core module
from bmcs_cross_section.core import BMCSModel, ui_field
from bmcs_cross_section.core import SymbolicModel
from bmcs_cross_section.core import get_ui_metadata, get_all_ui_fields

print("✅ Core module imports successful!")

## Test 1: Basic BMCSModel

Test creating a simple model with parameters and cached properties.

In [ ]:
class SimpleModel(BMCSModel):
    """Simple model for testing"""
    
    x: float = ui_field(
        1.0,
        label=r"$x$",
        range=(0.0, 10.0),
        description="First parameter"
    )
    
    y: float = ui_field(
        2.0,
        label=r"$y$",
        range=(0.0, 10.0),
        description="Second parameter"
    )
    
    @cached_property
    def z(self) -> float:
        """Computed property"""
        print(f"Computing z = x + y")
        return self.x + self.y

# Create instance
model = SimpleModel()
print(f"Model created: {model}")
print(f"z = {model.z}")  # Should print "Computing..." first time
print(f"z = {model.z}")  # Should NOT print "Computing..." (cached)

## Test 2: Update and Cache Invalidation

Test that updating parameters invalidates caches.

In [ ]:
print(f"Before update: z = {model.z}")

# Update parameter
model.update_params(x=5.0)
print(f"After update: z = {model.z}")  # Should recompute

# Verify value is correct
assert model.z == 7.0, "Cache invalidation failed!"
print("✅ Cache invalidation working correctly!")

## Test 3: UI Metadata Extraction

Test that UI metadata is correctly stored and retrieved.

In [ ]:
# Get all UI fields
ui_fields = get_all_ui_fields(model)
print(f"UI fields found: {list(ui_fields.keys())}")

# Check metadata for 'x'
x_metadata = get_ui_metadata(model, 'x')
print(f"\nMetadata for 'x':")
print(f"  Label: {x_metadata.label}")
print(f"  Range: {x_metadata.range}")
print(f"  Description: {x_metadata.description}")

assert x_metadata.label == r"$x$", "Label incorrect!"
assert x_metadata.range == (0.0, 10.0), "Range incorrect!"
print("\n✅ UI metadata extraction working!")

## Test 4: Model with Numpy Arrays

Test that numpy arrays work correctly with the model.

In [ ]:
class ArrayModel(BMCSModel):
    """Model with array computations"""
    
    a: float = ui_field(2.0, label="a", range=(0, 10))
    b: float = ui_field(3.0, label="b", range=(0, 10))
    
    def compute_line(self, x: np.ndarray) -> np.ndarray:
        """Compute y = a*x + b"""
        return self.a * x + self.b

# Test it
array_model = ArrayModel()
x = np.linspace(0, 10, 5)
y = array_model.compute_line(x)

print(f"x = {x}")
print(f"y = {y}")
print(f"Expected: {2*x + 3}")

assert np.allclose(y, 2*x + 3), "Array computation failed!"
print("\n✅ Numpy array support working!")

## Test 5: Symbolic Model Integration

Test the symbolic model functionality.

In [ ]:
from bmcs_cross_section.core.symbolic import SymbolicExpression
import sympy as sp

# Create symbolic model
symb = SymbolicModel()

# Define symbols
x = symb.symbol('x', real=True)
a = symb.symbol('a', real=True)
b = symb.symbol('b', real=True)

# Define expression: f(x) = a*x + b
expr = symb.expression('linear', a * x + b, ('x', 'a', 'b'))

# Test evaluation
result = expr(5.0, 2.0, 3.0)  # 2*5 + 3 = 13
print(f"f(5, 2, 3) = {result}")
assert result == 13.0, "Symbolic evaluation failed!"

# Test with arrays
x_array = np.array([0, 1, 2, 3, 4, 5])
y_array = expr(x_array, 2.0, 3.0)
print(f"f(x_array, 2, 3) = {y_array}")

expected = 2 * x_array + 3
assert np.allclose(y_array, expected), "Array evaluation failed!"
print("\n✅ Symbolic model working!")

## Test 6: Plot Function

Test creating a simple plot with the model.

In [ ]:
# Create a plot
fig, ax = plt.subplots(figsize=(8, 5))

x_plot = np.linspace(0, 10, 100)
y_plot = array_model.compute_line(x_plot)

ax.plot(x_plot, y_plot, 'b-', linewidth=2, label=f'y = {array_model.a}x + {array_model.b}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Simple Linear Function')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Plotting working!")

## Test 7: Validation

Test parameter validation with Pydantic.

In [ ]:
from pydantic import ValidationError

class ValidatedModel(BMCSModel):
    """Model with validation constraints"""
    
    # Positive value required
    E: float = ui_field(
        200000.0,
        label="E",
        range=(100000, 300000),
        description="Young's modulus",
        gt=0  # Must be greater than 0
    )
    
    # Range constraint
    f_y: float = ui_field(
        500.0,
        label="f_y",
        range=(400, 600),
        description="Yield strength",
        ge=400,
        le=600
    )

# Test valid creation
valid_model = ValidatedModel()
print(f"Valid model: {valid_model}")

# Test invalid value
try:
    invalid_model = ValidatedModel(E=-100)  # Negative, should fail
    print("❌ Validation failed to catch negative E!")
except ValidationError as e:
    print("✅ Validation caught negative E:")
    print(f"   {e.errors()[0]['msg']}")

# Test out of range
try:
    invalid_model = ValidatedModel(f_y=1000)  # Out of range
    print("❌ Validation failed to catch out of range f_y!")
except ValidationError as e:
    print("✅ Validation caught out of range f_y:")
    print(f"   {e.errors()[0]['msg']}")

## Test 8: Model Serialization

Test converting to/from dictionaries.

In [ ]:
# Convert to dict
model_dict = model.to_dict()
print(f"Model as dict: {model_dict}")

# Create from dict
new_model = SimpleModel.from_dict(model_dict)
print(f"New model from dict: {new_model}")

# Verify values match
assert new_model.x == model.x, "Deserialization failed!"
assert new_model.y == model.y, "Deserialization failed!"
print("\n✅ Serialization working!")

## Summary

All core module tests passed! ✅

### What Works:
1. ✅ Basic model creation with parameters
2. ✅ Cached properties with invalidation
3. ✅ UI metadata extraction
4. ✅ Numpy array support
5. ✅ Symbolic model integration
6. ✅ Plotting functionality
7. ✅ Parameter validation
8. ✅ Model serialization

### Next Steps:
- Test Jupyter UI adapter (next notebook)
- Test Streamlit adapter
- Create first material model (EC2 concrete)

## Cleanup

This notebook can be deleted or archived once core functionality is verified in production code.